# 🎙️ Whisper Voice Fine-Tuning (Google Colab 1-Click)
### Автоматическое дообучение нейросети OpenAI Whisper на вашем персональном голосе с экспортом в CTranslate2 (faster-whisper).

> **Инструкция**:
> 1. В верхнем меню выберите: `Среда выполнения` ➔ `Сменить тип среды выполнения` ➔ **T4 GPU**.
> 2. Нажмите `Среда выполнения` ➔ **Выполнить всё** (Ctrl+F9).
> 3. В Ячейке 3 выберите ваш файл `voice_dataset.zip`.
> 4. Через ~6 минут готовая модель автоматически скачается на ваш ПК!

In [ ]:
# @title 1. Проверка GPU акселератора
!nvidia-smi

In [ ]:
# @title 2. Установка зависимостей
!pip install -q --upgrade pip
!pip uninstall -y torchao
!pip install -q torch torchaudio transformers datasets peft accelerate evaluate ctranslate2 soundfile librosa

In [ ]:
# @title 3. Загрузка voice_dataset.zip
import os, zipfile
from google.colab import files

if not os.path.exists('voice_dataset.zip'):
    print('👉 Загрузите файл voice_dataset.zip:')
    uploaded = files.upload()

os.makedirs('dataset', exist_ok=True)
with zipfile.ZipFile('voice_dataset.zip', 'r') as zip_ref:
    zip_ref.extractall('dataset/')

print('✅ Датасет успешно распакован!')

In [ ]:
# @title 4. Запуск дообучения LoRA и CTranslate2 конвертации
!pip uninstall -y torchao
!rm -rf repo
!git clone https://github.com/yurac777/whisper-voice-dictation-ai.git repo

!python repo/scripts/fine_tune_voice.py \
    --data_dir dataset \
    --base_model openai/whisper-small \
    --output_dir /content/models/whisper-custom-voice \
    --ct2_output_dir /content/models/faster-whisper-custom-voice \
    --epochs 5 \
    --batch_size 4 \
    --learning_rate 1e-4 \
    --export_ct2

In [ ]:
# @title 5. Скачивание готовой модели на ваш компьютер
import shutil
from google.colab import files

shutil.make_archive('/content/faster-whisper-custom-voice', 'zip', '/content/models/faster-whisper-custom-voice')
print('🎉 Обучение успешно завершено! Скачиваем готовую модель...')
files.download('/content/faster-whisper-custom-voice.zip')